# Home-Work Stability Sensitivity Analysis

Two sensitivity checks for mobile phone visitation data:

1. **Temporal stability** — validates that home and work location assignments are behaviourally coherent: home presence peaks at night, work presence peaks during business hours on workdays.
2. **Sample representativeness** — log-log scatter of sample users per 1 km² grid cell vs. census population, with a fitted regression line and R²/p-value annotation.

All input paths and parameters are defined in the **Configuration** section below.

In [ ]:
import pandas as pd
import numpy as np
from shapely.geometry import Point
import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

## Configuration

Input paths and parameters are set in the next cell. All other cells run without modification.

### Expected input files

| Variable | Description | Required columns |
|----------|-------------|------------------|
| `USERS_FILE` | One row per user per month. | `user_id`, year, month, home H3 cell (res 9), work H3 cell (res 9) |
| `STAYS_FILE` | One row per detected stay. | `user_id`, year, month, day, hour (0–23), stay H3 cell (res 9) |
| `GRID_FILE` | 1 km² grid polygons with census population. | grid cell ID, population count, `geometry` |

In [ ]:
users = pd.read_parquet("scratch/data/user_home+work.parquet")

In [ ]:
users

In [ ]:
users = users[
    (users["YY"] == 2024) &
    (users["MM"].isin([3, 4, 5]))
].copy()

In [ ]:
stays = pd.read_parquet("scratch/data/stays_sufficient_users.parquet")

In [ ]:
stays = stays[
    (stays["YY"] == 2024) &
    (stays["MM"].isin([3, 4, 5]))
].copy()

In [ ]:
stays

In [ ]:
## Filtering only the cities

In [ ]:
import geopandas as gpd

# Load the study area polygon
gdf_helsinki = gpd.read_file("./data/Boundary_Helsinki.gpkg")  # or .shp

# Make sure it’s in WGS84 (lat/lon) for OSMnx
gdf_helsinki = gdf_helsinki.to_crs(epsg=4326)

In [ ]:
import geopandas as gpd

# Load the study area polygon
gdf = gpd.read_file("./data/Turku_region_boundary.geojson")  # or .shp

# Make sure it’s in WGS84 (lat/lon) for OSMnx
gdf = gdf.to_crs(epsg=4326)

In [ ]:
import geopandas as gpd

# Load the study area polygon
gdf_tampere = gpd.read_file("./data/Tampere_region_boundary.geojson")  # or .shp

# Make sure it’s in WGS84 (lat/lon) for OSMnx
gdf_tampere = gdf_tampere.to_crs(epsg=4326)

In [ ]:
import geopandas as gpd

# Load the study area polygon
gdf_oulu = gpd.read_file("./data/oulu_region_boundary.geojson")  # or .shp

# Make sure it’s in WGS84 (lat/lon) for OSMnx
gdf_oulu = gdf_oulu.to_crs(epsg=4326)

In [ ]:
import h3

geom = gdf_oulu.geometry.iloc[0]

oulu_hexes = set()

if geom.geom_type == "Polygon":
    coords = [(lat, lon) for lon, lat in geom.exterior.coords]
    oulu_hexes.update(
        h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
    )

elif geom.geom_type == "MultiPolygon":
    for poly in geom.geoms:
        coords = [(lat, lon) for lon, lat in poly.exterior.coords]
        oulu_hexes.update(
            h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
        )


In [ ]:
import h3

geom = gdf.geometry.iloc[0]

turku_hexes = set()

if geom.geom_type == "Polygon":
    coords = [(lat, lon) for lon, lat in geom.exterior.coords]
    turku_hexes.update(
        h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
    )

elif geom.geom_type == "MultiPolygon":
    for poly in geom.geoms:
        coords = [(lat, lon) for lon, lat in poly.exterior.coords]
        turku_hexes.update(
            h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
        )

In [ ]:
import h3

geom = gdf_tampere.geometry.iloc[0]

tampere_hexes = set()

if geom.geom_type == "Polygon":
    coords = [(lat, lon) for lon, lat in geom.exterior.coords]
    tampere_hexes.update(
        h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
    )

elif geom.geom_type == "MultiPolygon":
    for poly in geom.geoms:
        coords = [(lat, lon) for lon, lat in poly.exterior.coords]
        tampere_hexes.update(
            h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
        )


In [ ]:
import h3

geom = gdf_helsinki.geometry.iloc[0]

helsinki_hexes = set()

if geom.geom_type == "Polygon":
    coords = [(lat, lon) for lon, lat in geom.exterior.coords]
    helsinki_hexes.update(
        h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
    )

elif geom.geom_type == "MultiPolygon":
    for poly in geom.geoms:
        coords = [(lat, lon) for lon, lat in poly.exterior.coords]
        helsinki_hexes.update(
            h3.polyfill({"type": "Polygon", "coordinates": [coords]}, res=9)
        )


In [ ]:
users['helsinki_home'] = users['home_gid9'].isin(helsinki_hexes)

In [ ]:
users['turku_home'] = users['home_gid9'].isin(turku_hexes)


In [ ]:
users['tampere_home'] = users['home_gid9'].isin(tampere_hexes)

In [ ]:
users['oulu_home'] = users['home_gid9'].isin(oulu_hexes)

In [ ]:
users = users[
    (users['tampere_home']) |
    (users['turku_home']) |
    (users['helsinki_home']) |
    (users['oulu_home'])
].copy()

In [ ]:
stays= stays[
    stays['user_id'].isin(users['user_id'])
].copy()

In [ ]:
stays

In [ ]:
users

In [ ]:
grid_file = gpd.read_file("./data/1km_data/vaki2024_1km.shp")

In [ ]:
grid_file

In [ ]:
# --- FILE PATHS ----------------------------------------------------------------


# Path to the Parquet file containing one row per user per month.
USERS_FILE = users

# Path to the Parquet file containing individual stay records.
STAYS_FILE = stays

# Path to a GeoParquet (or Parquet) file with 1 km² grid polygons and census population.
GRID_FILE = grid_file

# --- COLUMN NAMES: users table ------------------------------------------------
COL_USER_ID    = "user_id"
COL_HOME_CELL  = "home_gid9"    # H3 index (res 9) for the home location
COL_WORK_CELL  = "work_gid9"    # H3 index (res 9) for the work location
COL_YEAR       = "YY"           # year (integer)
COL_MONTH      = "MM"           # month (integer, 1–12)

# --- COLUMN NAMES: stays table ------------------------------------------------
# user_id / year / month columns must share names with the users table above
COL_STAY_CELL  = "stay_gid9"    # H3 index (res 9) of the stay location
COL_DAY        = "DD"           # day-of-month (integer)
COL_HOUR       = "HH"           # hour of day (integer, 0–23)

# --- COLUMN NAMES: grid table -------------------------------------------------
COL_GRID_ID    = "grd_id"       # unique grid cell identifier
COL_POPULATION = "vaesto"       # census population count per cell
GRID_CRS       = "EPSG:3067"    # CRS of the grid geometries

# --- OPTIONAL TUNING ----------------------------------------------------------
# Days of the week counted as workdays (Monday = 0, Sunday = 6)
WORKDAY_DOW = set(range(0, 5))   # Mon–Fri

FIGURE_DPI = 150

## Data Loading

In [ ]:
grid  = grid_file
users = users
stays = stays

# Standardise to internal column names used throughout this notebook.
# If the columns are already named correctly these rename calls are no-ops.
users = users.rename(columns={
    COL_USER_ID:   'user_id',
    COL_HOME_CELL: 'home_gid9',
    COL_WORK_CELL: 'work_gid9',
    COL_YEAR:      'YY',
    COL_MONTH:     'MM',
})
stays = stays.rename(columns={
    COL_USER_ID:   'user_id',
    COL_STAY_CELL: 'stay_gid9',
    COL_YEAR:      'YY',
    COL_MONTH:     'MM',
    COL_DAY:       'DD',
    COL_HOUR:      'HH',
})
grid = grid.rename(columns={
    COL_GRID_ID:    'grd_id',
    COL_POPULATION: 'vaesto',
})

# Ensure grid is in the target CRS
if grid.crs is None:
    grid = grid.set_crs(GRID_CRS)
elif grid.crs.to_epsg() != int(GRID_CRS.split(':')[1]):
    grid = grid.to_crs(GRID_CRS)

print(f"Users: {len(users):,} rows | Stays: {len(stays):,} rows | Grid cells: {len(grid):,}")

## Section 1: Home-Work Temporal Stability

Each stay is classified as **home**, **work**, or **other** by comparing its H3 cell to the user's assigned home and work cells for the same month. Hourly presence probabilities are computed separately for workdays and weekends.

In [ ]:
# Keep only the columns needed for this section
stays_hw = stays[['user_id', 'YY', 'MM', 'DD', 'HH', 'stay_gid9']].copy()

# Join each stay with the user's home and work cell for the corresponding month
stays_hw = pd.merge(
    stays_hw,
    users[['user_id', 'YY', 'MM', 'home_gid9', 'work_gid9']],
    on=['user_id', 'YY', 'MM'],
    how='left'
)

# Classify workday vs weekend
stays_hw['date'] = pd.to_datetime(
    {
        'year':  pd.to_numeric(stays_hw['YY'], errors='coerce'),
        'month': pd.to_numeric(stays_hw['MM'], errors='coerce'),
        'day':   pd.to_numeric(stays_hw['DD'], errors='coerce'),
    },
    errors='coerce'
)
stays_hw['day_type'] = np.where(
    stays_hw['date'].dt.dayofweek.isin(WORKDAY_DOW), 'work_day', 'non_work_day'
)

# Classify each stay as home / work / other  (home wins when home cell == work cell)
stays_hw['HH'] = pd.to_numeric(stays_hw['HH'], errors='coerce')
is_home = stays_hw['stay_gid9'] == stays_hw['home_gid9']
is_work  = stays_hw['stay_gid9'] == stays_hw['work_gid9']
stays_hw['place_type'] = np.select(
    [is_home, is_work],
    ['home', 'work'],
    default='other'
)

In [ ]:
stays_hw

In [ ]:
# Hourly presence probabilities: P(place_type | hour, day_type)
hourly = (
    stays_hw.dropna(subset=['HH'])
            .groupby(['day_type', 'HH'])['place_type']
            .value_counts(normalize=True)
            .rename('prob')
            .reset_index()
            .pivot_table(
                index=['day_type', 'HH'],
                columns='place_type',
                values='prob',
                fill_value=0.0
            )
            .reset_index()
            .sort_values(['day_type', 'HH'])
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

colors  = {'work_day': 'tab:green', 'non_work_day': 'orange'}
markers = {'work_day': '^',         'non_work_day': 's'}

def place_hourly(place):
    df = hourly.copy()
    if place not in df.columns:
        df[place] = 0.0
    return (
        df[['HH', 'day_type', place]]
        .dropna(subset=['HH'])
        .assign(HH=lambda d: d['HH'].astype(int))
        .pivot_table(index='HH', columns='day_type', values=place, fill_value=0.0)
        .reindex(range(24), fill_value=0.0)
    )

for ax, place, title in zip(axes, ['home', 'work'], ['Home presence', 'Work presence']):
    ph = place_hourly(place)
    for dt in ['work_day', 'non_work_day']:
        if dt in ph.columns:
            ax.plot(
                ph.index, ph[dt],
                label='Workday' if dt == 'work_day' else 'Weekend',
                color=colors[dt], marker=markers[dt]
            )
    ax.set_title(title)
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('Probability')
    ax.set_xlim(0, 23)
    ax.set_xticks(range(0, 24, 3))
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==========================
# 3-hour presence profiles
# ==========================

# Create 3-hour bins:
# 0-2 -> 0, 3-5 -> 3, ..., 21-23 -> 21
stays_hw['HH_3h'] = (stays_hw['HH'] // 3) * 3

# Probability of being at home/work/other by 3-hour bin and day type
presence_3h = (
    stays_hw.dropna(subset=['HH_3h'])
            .groupby(['day_type', 'HH_3h'])['place_type']
            .value_counts(normalize=True)
            .rename('prob')
            .reset_index()
            .pivot_table(
                index=['day_type', 'HH_3h'],
                columns='place_type',
                values='prob',
                fill_value=0.0
            )
            .reset_index()
            .sort_values(['day_type', 'HH_3h'])
)

# Helper function
def place_3h(place):
    df = presence_3h.copy()

    if place not in df.columns:
        df[place] = 0.0

    return (
        df[['HH_3h', 'day_type', place]]
        .pivot_table(
            index='HH_3h',
            columns='day_type',
            values=place,
            fill_value=0.0
        )
        .reindex(range(0, 24, 3), fill_value=0.0)
    )

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors  = {'work_day': 'tab:green', 'non_work_day': 'orange'}
markers = {'work_day': '^',         'non_work_day': 's'}

bin_labels = [
    '0-2', '3-5', '6-8', '9-11',
    '12-14', '15-17', '18-20', '21-23'
]

for ax, place, title in zip(
    axes,
    ['home', 'work'],
    ['Home presence (3-hour bins)', 'Work presence (3-hour bins)']
):
    ph = place_3h(place)

    for dt in ['work_day', 'non_work_day']:
        if dt in ph.columns:
            ax.plot(
                ph.index,
                ph[dt],
                marker=markers[dt],
                color=colors[dt],
                linewidth=2,
                label='Workday' if dt == 'work_day' else 'Weekend'
            )

    ax.set_title(title)
    ax.set_xlabel('Time of day')
    ax.set_ylabel('Probability')
    ax.set_xticks(range(0, 24, 3))
    ax.set_xticklabels(bin_labels)
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

## Section 2: Sample Representativeness

Users are aggregated to their home 1 km² grid cells and compared to census population counts. A log-log regression quantifies how well the sample size scales with population.

In [ ]:
# Count unique users per home H3 cell (one row per user, regardless of month) 
users_unique = users.drop_duplicates(subset=['user_id'], keep='first') 



In [ ]:
# Count unique users per home H3 cell (one row per user, regardless of month)
users_unique = users.drop_duplicates(subset=['user_id'], keep='first')

home_counts = ( users_unique .dropna(subset=['home_gid9']).groupby('home_gid9').size().reset_index(name='user_count') ) 
home_counts = home_counts[home_counts['home_gid9'] != 'None']

def h3_to_point(idx):
    lat, lon = h3.h3_to_geo(idx)
    return Point(lon, lat)

home_counts['geometry'] = home_counts['home_gid9'].apply(h3_to_point)

home_counts = gpd.GeoDataFrame(
    home_counts,
    geometry='geometry',
    crs='EPSG:4326'
)

home_counts = home_counts.to_crs(GRID_CRS)

# Spatial join: assign each home-cell point to a grid polygon
joined = gpd.sjoin(
    home_counts,
    grid[['grd_id', 'geometry']],
    how='left',
    predicate='within'
)

# Sum user counts per grid cell and merge back
grid_user_counts = (
    joined.dropna(subset=['grd_id'])
          .groupby('grd_id')['user_count']
          .sum()
          .rename('users_home_count')
          .reset_index()
)
grid_with_users = (
    grid[['grd_id', 'vaesto', 'geometry']]
    .merge(grid_user_counts, on='grd_id', how='left')
)

print(f"Census population total : {grid['vaesto'].sum():,}")
print(f"Sample users total      : {home_counts['user_count'].sum():,}")
print(f"Grid cells with users   : {grid_with_users['users_home_count'].notna().sum():,} / {len(grid_with_users):,}")

In [ ]:
grid_plot = (
    grid_with_users[['users_home_count', 'vaesto']]
    .dropna()
    .query('users_home_count > 0 and vaesto > 0')
)

fig, ax = plt.subplots(figsize=(8, 8), dpi=FIGURE_DPI)

sns.scatterplot(
    data=grid_plot,
    x='users_home_count',
    y='vaesto',
    color='0.3',
    edgecolor='none',
    alpha=0.6,
    s=22,
    ax=ax
)
ax.set_xscale('log')
ax.set_yscale('log')

# Log-log regression
lx = np.log10(grid_plot['users_home_count'].values)
ly = np.log10(grid_plot['vaesto'].values)
res = linregress(lx, ly)

lx_line = np.linspace(lx.min(), lx.max(), 200)
ax.plot(
    10 ** lx_line,
    10 ** (res.intercept + res.slope * lx_line),
    color='red', lw=2, label='Fitted line'
)

r2 = res.rvalue ** 2
ax.text(
    0.03, 0.97,
    f'R\u00b2={r2:.3f}\np={res.pvalue:.2e}',
    transform=ax.transAxes,
    va='top', ha='left', fontsize=10,
    bbox=dict(boxstyle='round', fc='white', alpha=0.7, lw=0)
)

ax.set_xlabel('Users (home count, log scale)')
ax.set_ylabel('Population (census, log scale)')
ax.set_title('Grid-cell population vs. sample users (log\u2013log)')
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# -------------------------
# Filter low-population grid cells
# -------------------------

grid_plot = grid_with_users[['users_home_count', 'vaesto']].copy()

# Remove invalid / missing
grid_plot = grid_plot.dropna()

# Apply thresholds:
# - census population must be >= 5
# - users must be >= 1
grid_plot = grid_plot.query('vaesto >= 5 and users_home_count >= 1')

# -------------------------
# Plot
# -------------------------
fig, ax = plt.subplots(figsize=(8, 8), dpi=FIGURE_DPI)

sns.scatterplot(
    data=grid_plot,
    x='users_home_count',
    y='vaesto',
    color='0.3',
    edgecolor='none',
    alpha=0.6,
    s=22,
    ax=ax
)

ax.set_xscale('log')
ax.set_yscale('log')

# -------------------------
# Log-log regression
# -------------------------
lx = np.log10(grid_plot['users_home_count'].values)
ly = np.log10(grid_plot['vaesto'].values)

res = linregress(lx, ly)

lx_line = np.linspace(lx.min(), lx.max(), 200)
ax.plot(
    10 ** lx_line,
    10 ** (res.intercept + res.slope * lx_line),
    color='red',
    lw=2,
    label='Fitted line'
)

r2 = res.rvalue ** 2
ax.text(
    0.03, 0.97,
    f'R²={r2:.3f}\np={res.pvalue:.4e}',
    transform=ax.transAxes,
    va='top',
    ha='left',
    fontsize=10,
    bbox=dict(boxstyle='round', fc='white', alpha=0.7, lw=0)
)

ax.set_xlabel('Users')
ax.set_ylabel('Population')
ax.set_title('Grid-cell population vs. sample users (log–log)')

ax.legend(frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# Penetration per grid cell
grid_plot = grid_with_users[['users_home_count', 'vaesto']].copy()
grid_plot = grid_plot.dropna()
grid_plot = grid_plot.query('vaesto >= 5 and users_home_count >= 1')

grid_plot['penetration'] = grid_plot['users_home_count'] / grid_plot['vaesto']

# Overall statistics
mean_penetration = grid_plot['penetration'].mean()
median_penetration = grid_plot['penetration'].median()

print(f"Mean penetration:   {mean_penetration:.4f}")
print(f"Median penetration: {median_penetration:.4f}")